# Assignment: Building a Modular Data Sanitization & Exploration Engine

## Step 1: Install Required Libraries

In [2]:
!pip install plotly scipy scikit-learn --quiet

## Step 2: Import Libraries

In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from google.colab import files
import io

## Step 3: PlottingMethods Class

In [4]:
class PlottingMethods:
    """
    This class contains common chart drawing functions using Plotly.
    Each function creates a chart and returns it as HTML.
    """

    #Creates an interactive bar chart.
    def plot_bar(self, data, x_col, y_col, title='Bar Chart'):
        if data is None or data.empty: #if data is None or empty then this will run.
            print('No data available for bar chart.')
            return None
        fig = px.bar(data, x=x_col, y=y_col, title=title)
        return fig.to_html(full_html=False)

    #Creates a pie chart
    def plot_pie(self, data, names_col, values_col, title='Pie Chart'):
        if data is None or data.empty:#if data is None or empty then this will run.
            print('No data available for pie chart.')
            return None
        fig = px.pie(data, names=names_col, values=values_col, title=title)
        return fig.to_html(full_html=False)

    #Shows frequency distribution of a numeric column.
    def plot_histogram(self, data, col, title='Histogram'):
        if data is None or data.empty:
            print('No data available for histogram.')
            return None
        fig = px.histogram(data, x=col, title=title)
        return fig.to_html(full_html=False)

## Step 4: DataInspector Class

In [5]:
class DataInspector:
    """
    Main class used for:

      Loading datasets
      Cleaning data
      Handling missing values
      Detecting outliers
      Normalization
      Encoding
      Visualization
    """

    # ------------------------------------------------------------------ #
    #  1. DATA INGESTION & SANITIZATION
    # ------------------------------------------------------------------ #

    def __init__(self):
        self.df = None

    def upload_data(self):
        """
        This will allows user to upload CSV file
        This will automatically performs initial cleaning after loading.
        """

        uploaded = files.upload()
        for filename, content in uploaded.items():
            self.df = pd.read_csv(io.BytesIO(content))
            print(f'File "{filename}" uploaded successfully.')
        self._clean_and_convert()
        return self.df

    def load_from_url(self, url):
        """
        Load a CSV dataset directly from a public URL.
        Automatically cleans invalid values and corrects types.
        """

        self.df = pd.read_csv(url)
        print('Dataset loaded successfully.')
        self._clean_and_convert()
        return self.df

    def _clean_and_convert(self):
        """
        Replace garbage strings with NaN.
        """
        # Replace common garbage strings with NaN
        garbage = ['?', 'n/a', 'N/A', 'NA', 'NULL', 'null', '', ' ']
        self.df.replace(garbage, np.nan, inplace=True)

        # Auto-convert columns to numeric if conversion doesn't make them all-NaN
        for col in self.df.columns:
            if self.df[col].dtype == object:
                converted = pd.to_numeric(self.df[col], errors='coerce')
                if converted.notna().sum() > 0:   # at least one valid number
                    self.df[col] = converted

        print('Invalid values replaced and column types corrected.')

    # ------------------------------------------------------------------ #
    #  2. STRUCTURAL ANALYSIS & CLEANING
    # ------------------------------------------------------------------ #

    def data_summary(self):
        """
        Provides quick overview of dataset.

        This def Shows:
            Rows
            Columns
            Numeric columns
            Categorical columns
            Missing values
            First 20 rows
        """
        if self.df is None:
            print('No data loaded yet.')
            return

        print('=' * 50)
        print(f'Rows   : {self.df.shape[0]}')
        print(f'Columns: {self.df.shape[1]}')
        print('=' * 50)

        num_cols = self.df.select_dtypes(include=np.number).columns.tolist()
        cat_cols = self.df.select_dtypes(exclude=np.number).columns.tolist()
        print(f'Numeric columns ({len(num_cols)}): {num_cols}')
        print(f'Categorical columns ({len(cat_cols)}): {cat_cols}')
        print('=' * 50)

        print('Missing values per column:')
        print(self.df.isnull().sum())
        print('=' * 50)

        print('First 20 rows:')
        display(self.df.head(20))

    def handle_missing_values(self, column, strategy='mean', constant=None):
        """
        This def will fill missing values.
        """

        if self.df is None:
            print('No data loaded.')
            return None
        if column not in self.df.columns:
            print(f'Column "{column}" not found.')
            return self.df

        if strategy == 'mean':
            fill_val = self.df[column].mean()
        elif strategy == 'median':
            fill_val = self.df[column].median()
        elif strategy == 'mode':
            fill_val = self.df[column].mode()[0]
        elif strategy == 'constant':
            fill_val = constant
        else:
            print('Unknown strategy. Choose mean, median, mode, or constant.')
            return self.df

        self.df[column].fillna(fill_val, inplace=True)
        print(f'Missing values in "{column}" filled with {strategy} = {fill_val}.')
        return self.df

    def remove_duplicates(self):
        """
        This will remove repeated rows.
        """

        if self.df is None:
            print('No data loaded.')
            return None
        before = len(self.df)
        self.df.drop_duplicates(inplace=True)
        self.df.reset_index(drop=True, inplace=True)
        after = len(self.df)
        print(f'Removed {before - after} duplicate row(s). Rows remaining: {after}.')
        return self.df

    def handle_outliers(self, column, action='flag'):
        """
        Detect outliers using the IQR method.

        IQR = Q3 - Q1
        Lower Bound = Q1 - 1.5 * IQR
        Upper Bound = Q3 + 1.5 * IQR

        """
        if self.df is None:
            print('No data loaded.')
            return None
        if column not in self.df.columns:
            print(f'Column "{column}" not found.')
            return self.df

        Q1 = self.df[column].quantile(0.25)
        Q3 = self.df[column].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        outlier_mask = (self.df[column] < lower) | (self.df[column] > upper)
        outlier_count = outlier_mask.sum()
        print(f'Outliers detected in "{column}": {outlier_count}')
        print(f'Bounds -> Lower: {lower:.2f}, Upper: {upper:.2f}')

        if action == 'flag':
            display(self.df[outlier_mask])
        elif action == 'delete':
            self.df = self.df[~outlier_mask].reset_index(drop=True)
            print(f'{outlier_count} outlier row(s) removed.')
        return self.df

    def delete_rows(self):
        """
        User manually chooses rows to remove
        """

        if self.df is None:
            print('No data loaded.')
            return None
        user_input = input('Enter row indices to delete (comma-separated): ')
        indices = [int(i.strip()) for i in user_input.split(',') if i.strip().isdigit()]
        self.df.drop(index=indices, inplace=True, errors='ignore')
        self.df.reset_index(drop=True, inplace=True)
        print(f'Deleted rows: {indices}')
        return self.df

    def delete_columns(self):
        """
        User manually chooses columns to remove
        """

        if self.df is None:
            print('No data loaded.')
            return None
        user_input = input('Enter column names to delete (comma-separated): ')
        cols = [c.strip() for c in user_input.split(',')]
        self.df.drop(columns=cols, inplace=True, errors='ignore')
        print(f'Deleted columns: {cols}')
        return self.df

    # ------------------------------------------------------------------ #
    #  3. FEATURE ENGINEERING PREPARATION (NORMALIZATION)
    # ------------------------------------------------------------------ #

    def extract_normalized_numeric_data(self, method='minmax'):
        """
        Scale all numeric columns using the chosen method.

        Parameters:
            method (str): 'minmax', 'standard' (Z-score), or 'robust' (IQR-based).
        """
        if self.df is None:
            print('No data loaded.')
            return None

        num_df = self.df.select_dtypes(include=np.number).copy()

        scalers = {
            'minmax': MinMaxScaler(),
            'standard': StandardScaler(),
            'robust': RobustScaler()
        }
        if method not in scalers:
            print('Unknown method. Use minmax, standard, or robust.')
            return None

        scaler = scalers[method]
        scaled = scaler.fit_transform(num_df)
        result = pd.DataFrame(scaled, columns=num_df.columns)
        print(f'Numeric data scaled using "{method}" method.')
        return result

    def extract_normalized_categorical_data(self, method='onehot'):
        """
        Encode all categorical columns using the chosen method.

        Parameters:
            method (str): 'onehot', 'ordinal', or 'uniform' (scaled 0-1).

        """
        if self.df is None:
            print('No data loaded.')
            return None

        cat_df = self.df.select_dtypes(exclude=np.number).copy()
        if cat_df.empty:
            print('No categorical columns found.')
            return pd.DataFrame()

        # Fill NaN with 'Unknown' before encoding
        cat_df.fillna('Unknown', inplace=True)

        if method == 'onehot':
            enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
            encoded = enc.fit_transform(cat_df)
            col_names = enc.get_feature_names_out(cat_df.columns)
            result = pd.DataFrame(encoded, columns=col_names)

        elif method == 'ordinal':
            enc = OrdinalEncoder()
            encoded = enc.fit_transform(cat_df)
            result = pd.DataFrame(encoded, columns=cat_df.columns)

        elif method == 'uniform':
            # Map each category to a value between 0 and 1
            result = cat_df.copy()
            for col in result.columns:
                cats = result[col].unique()
                mapping = {c: i / (len(cats) - 1) if len(cats) > 1 else 0
                           for i, c in enumerate(cats)}
                result[col] = result[col].map(mapping)
        else:
            print('Unknown method. Use onehot, ordinal, or uniform.')
            return None

        print(f'Categorical data encoded using "{method}" method.')
        return result

    def get_merged_dataset(self, numeric_method='minmax', cat_method='ordinal'):
        """
        Combines:
          Processed numeric data
          Processed categorical data

        into one dataset.
        """
        num_df = self.extract_normalized_numeric_data(method=numeric_method)
        cat_df = self.extract_normalized_categorical_data(method=cat_method)

        if num_df is None:
            return cat_df
        if cat_df is None or cat_df.empty:
            return num_df

        merged = pd.concat([num_df.reset_index(drop=True),
                            cat_df.reset_index(drop=True)], axis=1)
        print('Merged dataset created successfully.')
        return merged

    # ------------------------------------------------------------------ #
    #  4. ADVANCED INTERACTIVE VISUALIZATION
    # ------------------------------------------------------------------ #

    def plot_univariate(self, column):
        """
        Generate a 3-panel subplot for a numeric column:
        Panel 1 - Horizontal Violin / Box plot
        Panel 2 - Scatter plot (Index vs Value)
        Panel 3 - Histogram
        """
        if self.df is None or column not in self.df.columns:
            print(f'Column "{column}" not found.')
            return

        series = self.df[column].dropna()

        fig = make_subplots(
            rows=1, cols=3,
            subplot_titles=('Violin / Box', 'Scatter (Index vs Value)', 'Histogram')
        )

        # Panel 1 – Violin
        fig.add_trace(
            go.Violin(y=series, name=column, box_visible=True, meanline_visible=True),
            row=1, col=1
        )
        # Panel 2 – Scatter
        fig.add_trace(
            go.Scatter(x=series.index, y=series, mode='markers', name=column),
            row=1, col=2
        )
        # Panel 3 – Histogram
        fig.add_trace(
            go.Histogram(x=series, name=column),
            row=1, col=3
        )

        fig.update_layout(title_text=f'Univariate Analysis: {column}', showlegend=False)
        fig.show()

    def plot_relationship(self, col_x, col_y):
        """
        Auto-detect column types and plot the appropriate chart:
        - Numeric vs Numeric  -> Scatter with OLS trendline
        - Categorical vs Numeric -> Box plot with data points
        - Categorical vs Categorical -> Grouped bar chart
        """
        if self.df is None:
            print('No data loaded.')
            return

        x_num = pd.api.types.is_numeric_dtype(self.df[col_x])
        y_num = pd.api.types.is_numeric_dtype(self.df[col_y])

        if x_num and y_num:
            # Numeric – Numeric: scatter with OLS trendline
            fig = px.scatter(self.df, x=col_x, y=col_y,
                             trendline='ols',
                             title=f'{col_x} vs {col_y} (OLS Trendline)')

        elif not x_num and y_num:
            # Categorical – Numeric: box plot
            fig = px.box(self.df, x=col_x, y=col_y,
                         points='all',
                         title=f'{col_x} vs {col_y} (Box Plot)')

        elif x_num and not y_num:
            # Numeric – Categorical: flip roles
            fig = px.box(self.df, x=col_y, y=col_x,
                         points='all',
                         title=f'{col_y} vs {col_x} (Box Plot)')

        else:
            # Categorical – Categorical: grouped bar chart
            ct = pd.crosstab(self.df[col_x], self.df[col_y]).reset_index()
            ct_melted = ct.melt(id_vars=col_x, var_name=col_y, value_name='Count')
            fig = px.bar(ct_melted, x=col_x, y='Count', color=col_y,
                         barmode='group',
                         title=f'{col_x} vs {col_y} (Grouped Bar)')

        fig.show()

    def plot_categorical_frequency(self, column):
        """
        Plot a bar chart showing raw counts and percentage labels
        for a categorical column.
        """
        if self.df is None or column not in self.df.columns:
            print(f'Column "{column}" not found.')
            return

        counts = self.df[column].value_counts().reset_index()
        counts.columns = [column, 'Count']
        counts['Percentage'] = (counts['Count'] / counts['Count'].sum() * 100).round(1)
        counts['Label'] = counts['Percentage'].astype(str) + '%'

        fig = px.bar(counts, x=column, y='Count',
                     text='Label',
                     title=f'Frequency of {column}')
        fig.update_traces(textposition='outside')
        fig.show()

    # ------------------------------------------------------------------ #
    #  5. DEEP STATISTICAL INSIGHTS – UNIFIED ASSOCIATION HEATMAP
    # ------------------------------------------------------------------ #

    def plot_all_associations_heatmap(self):
        """
        Creates one heatmap showing relationships between all columns.
        """
        if self.df is None:
            print('No data loaded.')
            return

        cols = self.df.columns.tolist()
        n = len(cols)
        matrix = pd.DataFrame(np.zeros((n, n)), index=cols, columns=cols)

        def cramers_v(x, y):
            """Compute Cramer's V statistic for two categorical columns."""
            ct = pd.crosstab(x, y)
            chi2 = stats.chi2_contingency(ct)[0]
            n_obs = ct.sum().sum()
            min_dim = min(ct.shape) - 1
            if min_dim == 0 or n_obs == 0:
                return 0.0
            return np.sqrt(chi2 / (n_obs * min_dim))

        def eta_squared(numeric_col, cat_col):
            """Compute Eta-squared between a numeric and categorical column."""
            groups = [group.dropna().values
                      for _, group in numeric_col.groupby(cat_col)]
            groups = [g for g in groups if len(g) > 0]
            if len(groups) < 2:
                return 0.0
            f_stat, p = stats.f_oneway(*groups)
            # Return magnitude only (0 to 1)
            ss_between = sum(len(g) * (g.mean() - numeric_col.mean())**2 for g in groups)
            ss_total = sum((numeric_col - numeric_col.mean())**2)
            return ss_between / ss_total if ss_total != 0 else 0.0

        for i, c1 in enumerate(cols):
            for j, c2 in enumerate(cols):
                if i == j:
                    matrix.iloc[i, j] = 1.0
                    continue
                s1 = self.df[c1].dropna()
                s2 = self.df[c2].dropna()
                common_idx = s1.index.intersection(s2.index)
                s1, s2 = s1[common_idx], s2[common_idx]

                num1 = pd.api.types.is_numeric_dtype(s1)
                num2 = pd.api.types.is_numeric_dtype(s2)

                if num1 and num2:
                    r, _ = stats.pearsonr(s1, s2)
                    matrix.iloc[i, j] = round(r, 3)
                elif not num1 and not num2:
                    matrix.iloc[i, j] = round(cramers_v(s1, s2), 3)
                else:
                    # Numeric vs Categorical
                    num_col = s1 if num1 else s2
                    cat_col = s2 if num1 else s1
                    eta = eta_squared(num_col, cat_col)
                    matrix.iloc[i, j] = round(eta, 3)

        fig = px.imshow(
            matrix,
            text_auto=True,
            color_continuous_scale='RdBu_r',
            zmin=-1, zmax=1,
            title='Unified Association Heatmap (Pearson r | Cramér\'s V | Eta²)'
        )
        fig.update_layout(width=900, height=700)
        fig.show()




## Step 5: Titanic Dataset
### Upload → Impute → Normalize → Visualize Associations

In [6]:
# --- Load Dataset from URL ---
inspector = DataInspector()
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = inspector.load_from_url(url)
df.head()

Dataset loaded successfully.
Invalid values replaced and column types corrected.


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,NaN,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,NaN,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,NaN,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803.0,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450.0,8.0500,NaN,S


In [7]:
# --- Dataset Summary ---
inspector.data_summary()

Rows   : 891
Columns: 12
Numeric columns (8): ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare']
Categorical columns (4): ['Name', 'Sex', 'Cabin', 'Embarked']
Missing values per column:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket         230
Fare             0
Cabin          687
Embarked         2
dtype: int64
First 20 rows:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,NaN,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,NaN,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,NaN,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803.0,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450.0,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877.0,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463.0,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909.0,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742.0,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736.0,30.0708,NaN,C


In [8]:
# --- Remove Duplicates ---
inspector.remove_duplicates()

Removed 0 duplicate row(s). Rows remaining: 891.


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,NaN,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,NaN,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,NaN,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803.0,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450.0,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536.0,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053.0,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,NaN,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369.0,30.0000,C148,C


In [10]:
# --- Handle Missing Values ---
inspector.handle_missing_values('Age', strategy='median')
inspector.handle_missing_values('Embarked', strategy='mode')
inspector.handle_missing_values('Fare', strategy='mean')

Missing values in "Age" filled with median = 28.0.
Missing values in "Embarked" filled with mode = S.
Missing values in "Fare" filled with mean = 32.204207968574636.


/tmp/ipykernel_995/1640028565.py:124: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  self.df[column].fillna(fill_val, inplace=True)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,NaN,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,NaN,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,NaN,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803.0,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450.0,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536.0,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053.0,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,28.0,1,2,NaN,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369.0,30.0000,C148,C


In [11]:
# --- Detect Outliers in Fare column ---
inspector.handle_outliers('Fare', action='flag')

Outliers detected in "Fare": 116
Bounds -> Lower: -26.72, Upper: 65.63


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,NaN,71.2833,C85,C
27,28,0,1,"Fortune, Mr. Charles Alexander",male,19.0,3,2,19950.0,263.0000,C23 C25 C27,S
31,32,1,1,"Spencer, Mrs. William Augustus (Marie Eugenie)",female,28.0,1,0,NaN,146.5208,B78,C
34,35,0,1,"Meyer, Mr. Edgar Joseph",male,28.0,1,0,NaN,82.1708,NaN,C
52,53,1,1,"Harper, Mrs. Henry Sleeper (Myna Haxtun)",female,49.0,1,0,NaN,76.7292,D33,C
...,...,...,...,...,...,...,...,...,...,...,...,...
846,847,0,3,"Sage, Mr. Douglas Bullen",male,28.0,8,2,NaN,69.5500,NaN,S
849,850,1,1,"Goldenberg, Mrs. Samuel L (Edwiga Grabowska)",female,28.0,1,0,17453.0,89.1042,C92,C
856,857,1,1,"Wick, Mrs. George Dennick (Mary Hitchcock)",female,45.0,1,1,36928.0,164.8667,NaN,S
863,864,0,3,"Sage, Miss. Dorothy Edith ""Dolly""",female,28.0,8,2,NaN,69.5500,NaN,S


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,NaN,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,NaN,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,NaN,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803.0,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450.0,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536.0,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053.0,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,28.0,1,2,NaN,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369.0,30.0000,C148,C


In [12]:
# --- Univariate Visualization for Age ---
inspector.plot_univariate('Age')

In [13]:
# --- Relationship Plots ---
inspector.plot_relationship('Age', 'Fare')
inspector.plot_relationship('Sex', 'Fare')
inspector.plot_relationship('Sex', 'Survived')

In [14]:
# --- Categorical Frequency Chart ---
inspector.plot_categorical_frequency('Pclass')
inspector.plot_categorical_frequency('Sex')

In [15]:
# --- Normalize Numeric Data ---
scaled_df = inspector.extract_normalized_numeric_data(method='minmax')
print(scaled_df.head())

Numeric data scaled using "minmax" method.
   PassengerId  Survived  Pclass       Age  SibSp  Parch    Ticket      Fare
0     0.000000       0.0     1.0  0.271174  0.125    0.0       NaN  0.014151
1     0.001124       1.0     0.0  0.472229  0.125    0.0       NaN  0.139136
2     0.002247       1.0     1.0  0.321438  0.000    0.0       NaN  0.015469
3     0.003371       1.0     0.0  0.434531  0.125    0.0  0.036480  0.103644
4     0.004494       0.0     1.0  0.434531  0.000    0.0  0.120221  0.015713


In [16]:
# --- Encode Categorical Data ---
encoded_df = inspector.extract_normalized_categorical_data(method='ordinal')
print(encoded_df.head())

Categorical data encoded using "ordinal" method.
    Name  Sex  Cabin  Embarked
0  108.0  1.0  147.0       2.0
1  190.0  0.0   81.0       0.0
2  353.0  0.0  147.0       2.0
3  272.0  0.0   55.0       2.0
4   15.0  1.0  147.0       2.0


In [17]:
# --- Merged Final Dataset ---
final_df = inspector.get_merged_dataset(numeric_method='minmax', cat_method='ordinal')
print(f'Final merged dataset shape: {final_df.shape}')
final_df.head()

Numeric data scaled using "minmax" method.
Categorical data encoded using "ordinal" method.
Merged dataset created successfully.
Final merged dataset shape: (891, 12)


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Ticket,Fare,Name,Sex,Cabin,Embarked
0,0.000000,0.0,1.0,0.271174,0.125,0.0,NaN,0.014151,108.0,1.0,147.0,2.0
1,0.001124,1.0,0.0,0.472229,0.125,0.0,NaN,0.139136,190.0,0.0,81.0,0.0
2,0.002247,1.0,1.0,0.321438,0.000,0.0,NaN,0.015469,353.0,0.0,147.0,2.0
3,0.003371,1.0,0.0,0.434531,0.125,0.0,0.036480,0.103644,272.0,0.0,55.0,2.0
4,0.004494,0.0,1.0,0.434531,0.000,0.0,0.120221,0.015713,15.0,1.0,147.0,2.0


In [18]:
# --- Unified Association Heatmap ---
# Using a subset of key columns for clarity
inspector.df = inspector.df[['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']]
inspector.plot_all_associations_heatmap()

## Step 6: PlottingMethods Demo